# HW5: Dyna-Q — Integrating Planning, Acting, and Learning

> - Full Name: **REDACTED**
> - Student ID: **REDACTED**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepRLCourse/Homework-5-Questions/blob/main/RL_HW5_Dyna.ipynb)
[![Open In kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/DeepRLCourse/Homework-5-Questions/main/RL_HW5_Dyna.ipynb)

## Overview

Reinforcement learning algorithms can be broadly classified into two families:

- **Model-free methods** (e.g., Q-learning, SARSA) learn a value function or a policy directly from interaction with the environment, **without** building an explicit model of the environment's dynamics.
- **Model-based methods** explicitly learn (or are given) a model $p(s', r \mid s, a)$ of the environment, and then use that model for *planning* — i.e., computing improved values or policies via simulated experience.

**Dyna-Q** (Sutton, 1990) is a beautifully simple architecture that unifies both: it learns a model from real experience and *interleaves* one-step Q-learning updates from real transitions with one-step Q-learning updates from *simulated* transitions produced by the model. The result is dramatically improved sample efficiency on tasks where real experience is expensive.

In this assignment you will:

1. Familiarize yourself with the [**Cliff Walking**](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) environment.
2. Implement $\varepsilon$-greedy and greedy policies.
3. Implement **Dyna-Q** for a deterministic environment.
4. Investigate how the number of planning steps affects learning.
5. Improve learning with **reward shaping**.
6. Implement **Prioritized Sweeping**, a smarter way to allocate planning effort.
7. (Bonus) Extend everything to the larger, stochastic **Taxi-v3** environment.

Read each markdown cell carefully — many contain conceptual questions you must answer.

In [ ]:
# @title Imports

import random
import numpy as np
import gymnasium as gym
from tqdm.notebook import trange
from heapq import heappush, heappop
from collections import defaultdict

import matplotlib
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec

import base64
import imageio
import IPython

import seaborn as sns

In [ ]:
# @title Visualization Functions

def embed_mp4(filename):
    video = open(filename, 'rb').read()
    b64 = base64.b64encode(video)
    tag = '''
    <video width="640" height="480" controls>
    <source src="data:video/mp4;base64,{0}" type="video/mp4">
    Your browser does not support the video tag.
    </video>'''.format(b64.decode())
    return IPython.display.HTML(tag)


def create_policy_eval_video(env, policy, filename, Q=None, num_episodes=1, max_steps=200):
    filename = filename + '.mp4'
    with imageio.get_writer(filename, fps=env.metadata.get('render_fps', 4)) as video:
        for _ in range(num_episodes):
            state, info = env.reset()
            video.append_data(env.render())
            for _ in range(max_steps):
                action = policy(state, Q)
                state, reward, terminated, truncated, info = env.step(action)
                video.append_data(env.render())
                if terminated or truncated:
                    break
    return embed_mp4(filename)


def plot_rewards(rewards, average_range=None, ax=None, show=False):
    if ax is None:
        fig, ax = plt.subplots()
    xs = range(1, len(rewards) + 1)
    ax.plot(xs, rewards, marker='.', linestyle='--', alpha=0.4, label='Episode Reward')
    ax.plot(xs, np.cumsum(rewards) / xs, label='Cumulative Average')
    if len(rewards) >= 20:
        window = min(50, len(rewards) // 5)
        smooth = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window, len(rewards) + 1), smooth, label=f'Rolling Mean (w={window})')
    ax.legend()
    ax.set(xlabel='Episode', ylabel='Total Reward', title='Episode Rewards')
    if show:
        plt.show()


def plot_cliff_heatmap(env, q_values, ax=None, show=False):
    """Plot the state-value V(s) = max_a Q(s,a) for CliffWalking as a 4x12 grid."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 3))
    rows, cols = 4, 12
    v = q_values.max(axis=1).reshape(rows, cols)
    best_a = q_values.argmax(axis=1)
    act_arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    labels = np.array([act_arrows[a] for a in best_a]).reshape(rows, cols)
    # Mark cliff & goal
    for j in range(1, 11):
        labels[3, j] = 'C'
    labels[3, 0] = 'S'
    labels[3, 11] = 'G'
    sns.heatmap(v, cmap='RdYlGn', annot=labels, fmt='s', ax=ax, cbar_kws={'label': 'V(s)'})
    ax.set_title('Greedy Policy & State Values')
    if show:
        plt.show()


def plot_performance(env, q_values, reward_sums):
    fig = plt.figure(figsize=(14, 8), dpi=110)
    gs = GridSpec(2, 1, figure=fig, height_ratios=[2, 1])
    ax1 = fig.add_subplot(gs[0, 0])
    plot_rewards(reward_sums, ax=ax1)
    ax2 = fig.add_subplot(gs[1, 0])
    plot_cliff_heatmap(env, q_values, ax=ax2)
    plt.tight_layout()
    plt.show()

# 1. Explore the Environment (5 points)

## The Cliff Walking Problem

**Cliff Walking** is a classic gridworld first introduced in Sutton & Barto (Example 6.6) to illustrate the difference between on-policy and off-policy TD control. The world is a $4 \times 12$ grid:

- The agent starts at the bottom-left corner **S**.
- The goal **G** is at the bottom-right corner.
- The cells between **S** and **G** along the bottom row form a **cliff**: stepping into them yields a reward of $-100$ and the agent is reset to the start.
- Every other transition yields a reward of $-1$.
- Actions: `0=Up, 1=Right, 2=Down, 3=Left`.
- The environment is **deterministic**.

Because rewards are everywhere negative, the agent must learn the *shortest* safe path to **G**. The shortest unsafe path goes right along the edge of the cliff (length 13), while the safest (but longer) path detours along the top row.

**Question 1.1.** Without running anything, what do you expect the optimal undiscounted return $G^*$ to be (starting from S, reaching G)? Show your reasoning.

`Your Answer:` We need 13 steps, so the best reward is -13.

In [ ]:
def make_cliff_env(render_mode=None):
    try:
        return gym.make(
            'CliffWalking-v1',
            render_mode=render_mode,
            is_slippery=False,
        )
    except (gym.error.UnregisteredEnv, gym.error.DeprecatedEnv):
        return gym.make('CliffWalking-v0', render_mode=render_mode)


env = make_cliff_env(render_mode='rgb_array')

print('Observation space:', env.observation_space)
print('Action space:', env.action_space)
print('Number of states:', env.observation_space.n)
print('Number of actions:', env.action_space.n)

Define a uniform random policy.

In [ ]:
def random_policy(*args):
    return random.randrange(env.action_space.n)

Visualize the random policy. (Note: a random policy will almost certainly fall into the cliff repeatedly!)

In [ ]:
create_policy_eval_video(env, random_policy, 'random_policy', num_episodes=2, max_steps=50)

# 2. Policies (5 points)

## Exploration vs. Exploitation

Any control algorithm must balance two competing pressures:

- **Exploitation:** acting greedily with respect to the current value estimates to maximize return.
- **Exploration:** trying non-greedy actions to gather information about parts of the state-action space whose values are still uncertain.

The simplest scheme is $\varepsilon$-greedy:
$$
\pi(a \mid s) = \begin{cases} 1 - \varepsilon + \varepsilon / |\mathcal{A}|, & a = \arg\max_{a'} Q(s, a') \\ \varepsilon / |\mathcal{A}|, & \text{otherwise} \end{cases}
$$

**Question 2.1.** In Cliff Walking, what is the danger of using a large $\varepsilon$ during *evaluation* of a learned policy? What is the danger of using too small an $\varepsilon$ during *learning*?

`Your Answer:` Like other areas of RL, too small epsilon will lead us to not exploring the solutions and too big epsilon will cause non stationary and instable learning.

In [ ]:
def greedy_policy(state: int, q_values: np.ndarray) -> int:
    return int(np.argmax(q_values[state]))

In [ ]:
def epsilon_greedy_policy(state: int, q_values: np.ndarray, epsilon: float) -> int:
    if np.random.random() < epsilon:
        return int(np.random.randint(q_values.shape[1]))
    return greedy_policy(state, q_values)

# 3. Dyna-Q (25 points)

## The Dyna Architecture

Dyna-Q maintains three things in parallel:

1. **A value function** $Q(s,a)$, updated by tabular one-step Q-learning.
2. **A model** of the environment. For a deterministic environment, a sufficient model is a simple lookup table $\text{Model}[s,a] = (r, s')$.
3. **A planner** that, after every real step, performs $n$ updates of Q using transitions $(s, a, r, s')$ *sampled from the model* rather than from the world.

The full update rules are:

**Direct RL (from real experience):**
$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha\Big[r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)\Big]
$$

**Model learning (deterministic):**
$$
\text{Model}[s_t, a_t] \leftarrow (r_t, s_{t+1})
$$

**Planning (repeat $n$ times):** sample a previously visited $(s, a)$, look up $(r, s')$ from the model, then
$$
Q(s, a) \leftarrow Q(s, a) + \alpha\Big[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\Big]
$$

**Question 3.1.** Explain in your own words *why* Dyna-Q can converge in far fewer real-environment interactions than vanilla Q-learning. What is the computational cost?

**Question 3.2.** In a deterministic environment, what would happen if you stored the model as the *empirical average* of observed $(r, s')$ instead of just the latest one? Would it matter here?

`Your Answers:`

3.1: Dyna-Q learns faster from real experience because each real transition is used for both a Q-learning update and simulated planning updates which improves sample efficiency, but it increases computation and requires extra memory to store the model.

3.2: Storing an empirical average would give almost the same result because each state-action pair always produces the same outcome. In a stochastic environment, the model should store probabilities or counts for all possible next states and rewards instead of only the latest transition.

## 3.1 Planning (10 points)

Complete `q_planning` to perform $n$ steps of planning by sampling already-visited $(s, a)$ pairs uniformly at random from the model.

In [ ]:
def q_planning(model: dict, q: np.ndarray, alpha: float, gamma: float, n: int) -> np.ndarray:
    if not model or n <= 0:
        return q

    states = list(model.keys())
    for _ in range(n):
        state = random.choice(states)
        action = random.choice(list(model[state].keys()))
        reward, next_state = model[state][action]

        td_target = reward + gamma * np.max(q[next_state])
        q[state, action] += alpha * (td_target - q[state, action])

    return q

## 3.2 Learning (15 points)

Now put everything together. At each real step you must:

1. Select an action with $\varepsilon$-greedy.
2. Step the real environment.
3. Perform the direct Q-learning update.
4. Update the model.
5. Perform $n$ planning steps.

Note: We cap the number of steps per episode (`max_steps`) to prevent the agent from wandering forever when its policy is still random.

In [ ]:
def dyna_q(n_episodes: int, env: gym.Env, epsilon: float, alpha: float,
           gamma: float, n: int, max_steps: int = 200) -> tuple[np.ndarray, np.ndarray]:

    reward_sums = np.zeros(n_episodes)
    q = np.zeros((env.observation_space.n, env.action_space.n))
    model = defaultdict(dict)

    for episode_i in (pbar := trange(n_episodes, leave=False)):
        state, info = env.reset()
        reward_sum, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = epsilon_greedy_policy(state, q, epsilon)

            next_state, reward, terminated, truncated, info = env.step(action)
            terminal = terminated or truncated

            bootstrap = 0.0 if terminal else gamma * np.max(q[next_state])
            td_target = reward + bootstrap
            q[state, action] += alpha * (td_target - q[state, action])

            model[state][action] = (reward, next_state)

            q_planning(model, q, alpha, gamma, n)

            state = next_state
            reward_sum += reward
            steps += 1

        pbar.set_description(f'Episode {episode_i}, R={reward_sum:.0f}')
        reward_sums[episode_i] = reward_sum

    return q, reward_sums

# 4. Experiments (15 points)

Run Dyna-Q with several values of the planning parameter $n$ (e.g. $n \in \{0, 5, 50\}$).
Note that **$n = 0$ recovers pure Q-learning**.

After running, answer the following:

**Question 4.1.** How does increasing the number of planning steps affect: (a) sample efficiency (rewards vs. episodes), and (b) wall-clock time per episode?

**Question 4.2.** Suppose the *first* successful trajectory occurs at episode $N_1$. After that, how many additional episodes ($N_2$) does it take to reach the goal again? Explain how $n$ affects each of $N_1$ and $N_2$, and *why* the effects differ.

**Question 4.3.** What happens to the *learned path* (cliff-edge vs. safer detour) as you vary $\varepsilon$? Why? Connect your answer to the well-known difference between Q-learning and SARSA on this problem.

**Question 4.4.** Cliff Walking is fully deterministic. Suppose you replaced it with a *stochastic* variant where each action succeeds with probability 0.8 and otherwise slips perpendicular. What failure modes would the current **deterministic** Dyna-Q exhibit? Propose how you would modify the model to fix this.

`Your Answers:`

4.1: Planning improved performance compared with normal Q-learning, but increasing the planning from 5 to 50 did not provide much benefit, because of computation. Therefore, 5 planning steps were more cost-effective in this experiment.

4.2: With seed 2025, all settings reached the goal for the first time in episode 1. The second success occurred in episode 4 for n = 0 and episode 2 for n = 5 and n = 50. Planning has little effect before the goal is first discovered, but after discovery it helps as a signal for useful information and repeat successful behavior sooner.

4.3: A larger epsilon causes more random actions, making more cliff falls and reducing the return. Q-learning is off-policy, so it usually learns the shortest path near the cliff, while SARSA accounts for exploratory actions and often learns a longer but safer route.

4.4: Would make the model inaccurate because earlier possible outcomes would be overwritten. This could create unstable values and risky policies. The model should instead store transition probabilities or something like frequencies for every possible outcome.

In [ ]:
np.random.seed(2025)
random.seed(2025)

params = {
    'epsilon': 0.1,    # epsilon-greedy exploration rate
    'alpha':   0.5,    # learning rate (tabular, deterministic env => can be large)
    'gamma':   0.99,   # discount factor
    'n':       10,     # number of planning steps per real step
}

n_episodes = 500

env = make_cliff_env()

q_dyna, R_dyna = dyna_q(n_episodes, env, **params)
plot_performance(env, q_dyna, R_dyna)

**Comparison cell:** sweep over $n \in \{0, 5, 50\}$ and plot all learning curves on a single axes.

In [ ]:
planning_steps = [0, 5, 50]
results_by_n = {}

fig, ax = plt.subplots(figsize=(10, 5))
window = 50

for n_plan in planning_steps:
    np.random.seed(2025)
    random.seed(2025)

    env_n = make_cliff_env()
    q_n, rewards_n = dyna_q(
        n_episodes,
        env_n,
        epsilon=params['epsilon'],
        alpha=params['alpha'],
        gamma=params['gamma'],
        n=n_plan,
    )
    results_by_n[n_plan] = {'q': q_n, 'rewards': rewards_n}

    if len(rewards_n) >= window:
        rolling = np.convolve(rewards_n, np.ones(window) / window, mode='valid')
        episodes = np.arange(window, len(rewards_n) + 1)
    else:
        rolling = rewards_n
        episodes = np.arange(1, len(rewards_n) + 1)

    ax.plot(episodes, rolling, label=f'n={n_plan}')
    env_n.close()

ax.set(
    xlabel='Episode',
    ylabel=f'Rolling-mean return (window={window})',
    title='Effect of the Dyna-Q Planning Budget',
)
ax.legend(title='Planning steps')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# 5. Improving Performance (15 points)

Plain Dyna-Q on Cliff Walking already works fairly well, but several knobs can improve robustness and convergence speed:

1. **Optimistic initialization.** Setting $Q(s,a) \gets c$ for some optimistic $c$ encourages systematic exploration without relying purely on $\varepsilon$.
2. **Decaying $\varepsilon$.** Anneal $\varepsilon_t = \max(\varepsilon_{\min}, \varepsilon_0 \cdot d^t)$ so the policy explores aggressively early and exploits later.
3. **Decaying $\alpha$.** Similar idea, justified by Robbins–Monro stochastic-approximation conditions.
4. **Boltzmann (softmax) exploration** instead of $\varepsilon$-greedy.
5. **Adaptive planning budget.** Increase $n$ only after the agent has accumulated enough model entries.

Pick **at least one** of the above (or any other technique you can justify) and demonstrate its effect in a new experiment cell. **Do not overwrite your previous experiments** — leave them for comparison.

**Question 5.1.** Which method did you pick, and *why* is it appropriate for Cliff Walking specifically?

`Your Answer:` I used an exponentially decaying epsilon so that the agent explored more at the beginning and became more greedy later. Early performance was poor because of frequent exploration, but the final 50-episode average improved to about -15.22, which is close to the optimal return of -13.

In [ ]:
def dyna_q_epsilon_decay(n_episodes: int, env: gym.Env, epsilon_start: float,
                         epsilon_min: float, epsilon_decay: float, alpha: float,
                         gamma: float, n: int,
                         max_steps: int = 200) -> tuple[np.ndarray, np.ndarray]:

    reward_sums = np.zeros(n_episodes)
    q = np.zeros((env.observation_space.n, env.action_space.n))
    model = defaultdict(dict)

    for episode_i in (pbar := trange(n_episodes, leave=False)):
        epsilon = max(epsilon_min, epsilon_start * (epsilon_decay ** episode_i))
        state, info = env.reset()
        reward_sum, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = epsilon_greedy_policy(state, q, epsilon)
            next_state, reward, terminated, truncated, info = env.step(action)
            terminal = terminated or truncated

            bootstrap = 0.0 if terminal else gamma * np.max(q[next_state])
            td_target = reward + bootstrap
            q[state, action] += alpha * (td_target - q[state, action])

            model[state][action] = (reward, next_state)
            q_planning(model, q, alpha, gamma, n)

            state = next_state
            reward_sum += reward
            steps += 1

        pbar.set_description(
            f'Episode {episode_i}, epsilon={epsilon:.3f}, R={reward_sum:.0f}'
        )
        reward_sums[episode_i] = reward_sum

    return q, reward_sums


np.random.seed(2025)
random.seed(2025)
env_decay = make_cliff_env()

q_decay, R_decay = dyna_q_epsilon_decay(
    n_episodes=n_episodes,
    env=env_decay,
    epsilon_start=0.30,
    epsilon_min=0.01,
    epsilon_decay=0.985,
    alpha=params['alpha'],
    gamma=params['gamma'],
    n=params['n'],
)

# Compare against the fixed-epsilon baseline from Section 4.
fig, ax = plt.subplots(figsize=(10, 5))
window = 50
for label, rewards in [
    (f"Fixed epsilon={params['epsilon']}", R_dyna),
    ('Decaying epsilon: 0.30 -> 0.01', R_decay),
]:
    if len(rewards) >= window:
        rolling = np.convolve(rewards, np.ones(window) / window, mode='valid')
        episodes = np.arange(window, len(rewards) + 1)
    else:
        rolling = rewards
        episodes = np.arange(1, len(rewards) + 1)
    ax.plot(episodes, rolling, label=label)

ax.set(
    xlabel='Episode',
    ylabel=f'Rolling-mean return (window={window})',
    title='Dyna-Q: Fixed vs. Decaying Exploration',
)
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plot_performance(env_decay, q_decay, R_decay)
env_decay.close()

# 6. Reward Shaping (10 points)

**Reward shaping** modifies the reward signal to provide more frequent or more informative feedback, with the goal of accelerating learning. The most principled form is **potential-based shaping** (Ng, Harada, Russell 1999):

$$
F(s, a, s') = \gamma \Phi(s') - \Phi(s)
$$

where $\Phi : \mathcal{S} \to \mathbb{R}$ is any potential function. A foundational theorem states:

> *Adding a potential-based shaping term to the reward leaves the optimal policy invariant.*

Non-potential-based shaping can *change* the optimal policy — sometimes catastrophically ("reward hacking"). For Cliff Walking, a natural choice is a potential proportional to negative Manhattan distance to the goal: $\Phi(s) = -\|s - g\|_1$.

**Question 6.1.** Why does potential-based shaping preserve the optimal policy? Sketch the argument.

**Question 6.2.** Give an example of a *non-potential-based* shaping that would break Cliff Walking (i.e., cause the agent to learn a suboptimal policy).

`Your Answers:`

6.1: The potential terms are canceled over a trajectory, and at the end we have a constant. So we know this constant would not change the optimal policy.

6.2: A harmful shaping reward would be giving a positive bonus whenever the agent moves left. The agent might then repeatedly move left and right to collect extra rewards instead of reaching the goal. This changes the optimal policy and creates reward hacking.

In [ ]:
from gymnasium.envs.toy_text import CliffWalkingEnv

class ShapedCliffWalkingEnv(CliffWalkingEnv):

    GOAL_STATE = 47
    NROWS, NCOLS = 4, 12

    def __init__(self, gamma: float = 0.99, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self._prev_state = None

    def reset(self, **kwargs):
        obs, info = super().reset(**kwargs)
        self._prev_state = obs
        return obs, info

    def potential(self, state: int) -> float:
        row, col = divmod(int(state), self.NCOLS)
        goal_row, goal_col = divmod(self.GOAL_STATE, self.NCOLS)
        return -float(abs(row - goal_row) + abs(col - goal_col))

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)

        shaping = (
            self.gamma * self.potential(obs)
            - self.potential(self._prev_state)
        )
        shaped_reward = float(reward + shaping)

        self._prev_state = obs
        return obs, shaped_reward, terminated, truncated, info

In [ ]:
np.random.seed(2025)
random.seed(2025)

params = {
    'epsilon': 0.1,
    'alpha':   0.5,
    'gamma':   0.99,
    'n':       10,
}
n_episodes = 500

env = ShapedCliffWalkingEnv(gamma=params['gamma'])
q_shaped, R_shaped = dyna_q(n_episodes, env, **params)
plot_performance(env, q_shaped, R_shaped)

**Question 6.3.** Compare the learning curves with and without shaping. Did shaping help in the *early* episodes? In the *late* episodes? Why?

`Your Answer:` Reward shaping was most useful early in training because it gave immediate feedback about whether the agent was moving closer to the goal. Later, both shaped and unshaped methods learned the same optimal 13-step path, so the advantage became smaller.

# 7. Prioritized Sweeping (20 points)

Uniform sampling of $(s, a)$ pairs in the planning loop is wasteful. Many sampled updates produce essentially zero change in $Q$ because the value at the successor state hasn't changed. **Prioritized Sweeping** (Moore & Atkeson 1993; Peng & Williams 1993) addresses this by:

1. Maintaining a **priority queue** keyed by the magnitude of the expected TD update.
2. On each real step, computing the TD error $\delta$ for the current $(s, a)$. If $|\delta| > \theta$ (a threshold), push $(s, a)$ onto the queue with priority $|\delta|$.
3. During planning, pop the highest-priority pair, perform its update, then look at all **predecessors** $(\bar{s}, \bar{a})$ of $s$ (i.e., pairs known by the model to lead to $s$). Compute *their* TD errors; if they exceed $\theta$, push them too.

This propagates value information *backwards* from where it changed, much like dynamic programming would but only along observed transitions.

**Question 7.1.** Why is the magnitude of the TD error a sensible priority? What property does it estimate?

**Question 7.2.** Python's `heapq` is a *min-heap*. How do you use it as a *max-heap* for priorities?

**Question 7.3.** What is the role of $\theta$? What happens if $\theta = 0$? If $\theta$ is very large?

`Your Answers:`

7.1: The TD error is a useful priority because it measures how different the current Q-value is from its Bellman target. A large TD error means the update may significantly change the value estimate and should therefore be processed earlier.

7.2: Negative priorities are stored to simulate a max-heap. The transition with the largest original priority has the smallest negative value and is removed first.

7.3: The theta prevents very small TD errors from entering the queue. If it is zero, many insignificant updates are performed, increasing computation. If theta is too large, useful updates may be ignored, so a small positive threshold provides a good balance.

## 7.1 Priority Planning (10 points)

Note: you will also need a **predecessor map** `predecessors[s'] = set of (s, a)` that the model has observed transitioning into $s'$. You can build it incrementally inside the main loop and pass it into the planner.

In [ ]:
def q_planning_priority(model: dict, predecessors: dict, q: np.ndarray,
                        priorities: list, alpha: float, gamma: float,
                        n: int, theta: float) -> np.ndarray:

    for _ in range(n):
        if not priorities:
            break

        _, (state, action) = heappop(priorities)

        if state not in model or action not in model[state]:
            continue

        reward, next_state = model[state][action]
        td_target = reward + gamma * np.max(q[next_state])
        q[state, action] += alpha * (td_target - q[state, action])

        for prev_state, prev_action in predecessors.get(state, set()):
            prev_reward, _ = model[prev_state][prev_action]
            prev_delta = (
                prev_reward
                + gamma * np.max(q[state])
                - q[prev_state, prev_action]
            )
            if abs(prev_delta) > theta:
                heappush(
                    priorities,
                    (-abs(prev_delta), (prev_state, prev_action)),
                )

    return q

## 7.2 Learning with Prioritized Sweeping (10 points)

In [ ]:
def dyna_q_priority(n_episodes: int, env: gym.Env, epsilon: float, alpha: float,
                   gamma: float, n: int, theta: float,
                   max_steps: int = 200) -> tuple[np.ndarray, np.ndarray]:

    reward_sums = np.zeros(n_episodes)
    q = np.zeros((env.observation_space.n, env.action_space.n))
    model = defaultdict(dict)
    predecessors = defaultdict(set)
    priorities = []

    for episode_i in (pbar := trange(n_episodes, leave=False)):
        state, info = env.reset()
        reward_sum, terminal, steps = 0.0, False, 0

        while not terminal and steps < max_steps:
            action = epsilon_greedy_policy(state, q, epsilon)

            next_state, reward, terminated, truncated, info = env.step(action)
            terminal = terminated or truncated

            bootstrap = 0.0 if terminal else gamma * np.max(q[next_state])
            td_target = reward + bootstrap
            td_error = td_target - q[state, action]
            q[state, action] += alpha * td_error

            if action in model[state]:
                old_next_state = model[state][action][1]
                if old_next_state != next_state:
                    predecessors[old_next_state].discard((state, action))

            model[state][action] = (reward, next_state)
            predecessors[next_state].add((state, action))

            if abs(td_error) > theta:
                heappush(priorities, (-abs(td_error), (state, action)))

            q_planning_priority(
                model,
                predecessors,
                q,
                priorities,
                alpha,
                gamma,
                n,
                theta,
            )

            state = next_state
            reward_sum += reward
            steps += 1

        pbar.set_description(f'Episode {episode_i}, R={reward_sum:.0f}')
        reward_sums[episode_i] = reward_sum

    return q, reward_sums

# 8. Final Experiments (5 points)

Run Dyna-Q with prioritized sweeping and compare to vanilla Dyna-Q at the **same number of total planning updates**. Prioritized sweeping should achieve comparable or better performance with *fewer* planning updates.

In [ ]:
np.random.seed(2025)
random.seed(2025)

params = {
    'epsilon': 0.1,
    'alpha':   0.5,
    'gamma':   0.99,
    'n':       10,
    'theta':   1e-3,
}
n_episodes = 500

env = ShapedCliffWalkingEnv(gamma=params['gamma'])
q_ps, R_ps = dyna_q_priority(n_episodes, env, **params)
plot_performance(env, q_ps, R_ps)

Visualize the learned greedy policy.

In [ ]:
env_vis = ShapedCliffWalkingEnv(gamma=params['gamma'], render_mode='rgb_array')
create_policy_eval_video(env_vis, greedy_policy, 'DynaQ_PS', Q=q_ps, max_steps=50)

**Question 8.1.** Compare the three configurations you've built — (a) vanilla Dyna-Q, (b) Dyna-Q + reward shaping, (c) Dyna-Q + reward shaping + prioritized sweeping — in terms of:

- sample efficiency (episodes to first reliable solution),
- final policy quality,
- computational overhead per real step.

Which is the most cost-effective in your view, and why?

`Your Answer:` Simple Dyna-Q improved learning by uniform planning, while reward shaping gave better early guidance toward the goal. Prioritized sweeping focused updates on transitions with large TD errors but required extra queue and predecessor operations. All methods learned the same optimal 13-step path, and shaping with prioritized sweeping was the most sample-efficient overall, although its advantage was not too recognizable in this small environment.